# Purpose :

take data from /data, and preprocess it in /data_prepared

In [17]:
# Imports
import pandas as pd
import os

abbrev = pd.read_csv('./data/abbrev.csv')
awardsCoaches = pd.read_csv('./data/AwardsCoaches.csv')
awardsMisc = pd.read_csv('./data/AwardsMisc.csv')
awardsPlayers = pd.read_csv('./data/AwardsPlayers.csv')
coaches = pd.read_csv('./data/Coaches.csv')
combinedShutouts = pd.read_csv('./data/CombinedShutouts.csv')
goalies = pd.read_csv('./data/Goalies.csv')
goaliesSC = pd.read_csv('./data/GoaliesSC.csv')
goaliesShootout = pd.read_csv('./data/GoaliesShootout.csv')
hof = pd.read_csv('./data/HOF.csv')
master = pd.read_csv('./data/Master.csv')
scoring = pd.read_csv('./data/Scoring.csv')
scoringSC = pd.read_csv('./data/ScoringSC.csv')
scoringShootout = pd.read_csv('./data/ScoringShootout.csv')
scoringSup = pd.read_csv('./data/ScoringSup.csv')
seriesPost = pd.read_csv('./data/SeriesPost.csv')
teams = pd.read_csv('./data/Teams.csv')
teamsHalf = pd.read_csv('./data/TeamsHalf.csv')
teamSplits = pd.read_csv('./data/TeamSplits.csv')
teamsPost = pd.read_csv('./data/TeamsPost.csv')
teamsSC = pd.read_csv('./data/TeamsSC.csv')
teamsVsTeam = pd.read_csv('./data/TeamVsTeam.csv')

# Select Data

We only keep years above 1979 (year in which there is only NHL league)

In [18]:
YEAR = 1979

awardsCoaches       = awardsCoaches[awardsCoaches['year'] > YEAR]
awardsMisc          = awardsMisc[awardsMisc['year'] > YEAR]
awardsPlayers       = awardsPlayers[awardsPlayers['year'] > YEAR]
coaches             = coaches[coaches['year'] > YEAR]
combinedShutouts    = combinedShutouts[combinedShutouts['year'] > YEAR]
goalies             = goalies[goalies['year'] > YEAR]
goaliesSC           = goaliesSC[goaliesSC['year'] > YEAR]
goaliesShootout     = goaliesShootout[goaliesShootout['year'] > YEAR]
hof                 = hof[hof['year'] > YEAR]
scoring             = scoring[scoring['year'] > YEAR]
scoringSC           = scoringSC[scoringSC['year'] > YEAR]
scoringShootout     = scoringShootout[scoringShootout['year'] > YEAR]
scoringSup          = scoringSup[scoringSup['year'] > YEAR]
seriesPost          = seriesPost[seriesPost['year'] > YEAR]
teams               = teams[teams['year'] > YEAR]
teamsHalf           = teamsHalf[teamsHalf['year'] > YEAR]
teamSplits          = teamSplits[teamSplits['year'] > YEAR]
teamsPost           = teamsPost[teamsPost['year'] > YEAR]
teamsSC             = teamsSC[teamsSC['year'] > YEAR]
teamsVsTeam         = teamsVsTeam[teamsVsTeam['year'] > YEAR]

# Clean Data

Remove the tables :
- TeamsSC
- GoaliesSC
- TeamsHalf
- ScoringSC
- AwardsMisc
- ScoringSup

In [19]:
awardsMisc  = None
goaliesSC   = None
scoringSC   = None
scoringSup  = None
teamsHalf   = None
teamsSC     = None

maybe ?
- GoaliesShootout 
- ScoringShootout 

In [20]:
# goaliesShootout = None
# scoringShootout = None

# Construct Data

Add a column for teams which have just changed coach (1 if so, 0 else)

In [21]:
if 'coach_changed' in teams.columns:
    teams = teams.drop(columns=['coach_changed'])

coach_changes = coaches.groupby(['tmID', 'year'])['stint'].max().reset_index()
coach_changes['coach_changed'] = (coach_changes['stint'] > 1).astype(int)
teams = pd.merge(teams, coach_changes[['tmID', 'year', 'coach_changed']], on=['tmID', 'year'], how='left')
teams['coach_changed'] = teams['coach_changed'].fillna(0).astype(int)

teams_print = teams[['tmID', 'year', 'coach_changed']]

print("Changed coaches :")
print(len(teams_print[teams_print['coach_changed'] == 1]))
print()
print("Didn't change coaches :")
print(len(teams_print[teams_print['coach_changed'] == 0]))
print()

Changed coaches :
140

Didn't change coaches :
592



Add a column for teams that shows the global ranking (1-15) in the current conference (Eastern or Western depending on the team).
It computes rankings from the divisions to build a per-conference ranking.

In [22]:
teams = teams.sort_values(['year', 'confID', 'Pts', 'tmID'],
                          ascending=[True, True, False, True]).copy()
teams['conference_rank'] = (
    teams.groupby(['year', 'confID']).cumcount() + 1
)
teams['conference_rank'] = teams['conference_rank'].astype(int)

teams = teams.sort_values(['tmID', 'year'])
teams['prev_conference_rank'] = teams.groupby('tmID')['conference_rank'].shift(1)

YEAR_TO_DISPLAY = 2008

print(f"Conference ranking for {YEAR_TO_DISPLAY}:")
print(
    teams[teams['year'] == YEAR_TO_DISPLAY]
    [['prev_conference_rank', 'conference_rank', 'confID', 'tmID', 'name', 'Pts']]
    .sort_values(['confID', 'conference_rank'])
    .to_string(index=False)
)

Conference ranking for 2008:
 prev_conference_rank  conference_rank confID tmID                  name  Pts
                  6.0                1     EC  BOS         Boston Bruins  116
                  8.0                2     EC  WAS   Washington Capitals  108
                  3.0                3     EC  NJD     New Jersey Devils  106
                  5.0                4     EC  PHI   Philadelphia Flyers   99
                  2.0                5     EC  PIT   Pittsburgh Penguins   99
                  9.0                6     EC  CAR   Carolina Hurricanes   97
                  4.0                7     EC  NYR      New York Rangers   95
                 11.0                8     EC  FLO      Florida Panthers   93
                  1.0                9     EC  MTL    Montreal Canadiens   93
                 10.0               10     EC  BUF        Buffalo Sabres   91
                  7.0               11     EC  OTT       Ottawa Senators   83
                 12.0              

# Integrate Data

In [23]:
# Not sure what to do here yet ...

# Format Data

Normalize the data

In [24]:
from sklearn.preprocessing import MinMaxScaler
import os

EXCLUDE_COLS = {
    'tmID', 'year', 'playerID', 'coachID', 'confID', 'divID',
    'lgID', 'teamID', 'franchID', 'pos', 'type', 'award',
    'coach_changed',
}

def normalize_df(df):
    """Applies a MinMax scaling [0,1] on nums columns (except IDs)."""
    if df is None or df.empty:
        return df
    df = df.copy()
    numeric_cols = df.select_dtypes(include='number').columns
    cols_to_norm = [c for c in numeric_cols if c not in EXCLUDE_COLS]
    if cols_to_norm:
        scaler = MinMaxScaler()
        df[cols_to_norm] = scaler.fit_transform(df[cols_to_norm])
    return df

tables = {
    'abbrev':           abbrev,
    'awardsCoaches':    awardsCoaches,
    'awardsPlayers':    awardsPlayers,
    'coaches':          coaches,
    'combinedShutouts': combinedShutouts,
    'goalies':          goalies,
    'goaliesShootout':  goaliesShootout,
    'hof':              hof,
    'master':           master,
    'scoring':          scoring,
    'scoringShootout':  scoringShootout,
    'seriesPost':       seriesPost,
    'teams':            teams,
    'teamSplits':       teamSplits,
    'teamsPost':        teamsPost,
    'teamsVsTeam':      teamsVsTeam,
}

normalized_tables = {name: normalize_df(df) for name, df in tables.items()}

C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\_array_api.py:794: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\_array_api.py:814: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\_array_api.py:794: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\_array_api.py:814: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\_array_api.py:794: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\utils

# Save

In [25]:
os.makedirs('./data_prepared', exist_ok=True)

for name, df in normalized_tables.items():
    if df is None or df.empty:
        continue
    path = f'./data_prepared/{name}.csv'
    df.to_csv(path, index=False)
    print(f"  saved {name:20s} -> {path}  ({len(df)} rows, {len(df.columns)} cols)")

print("\nAll tables saved to ./data_prepared/")

  saved abbrev               -> ./data_prepared/abbrev.csv  (58 rows, 3 cols)
  saved awardsCoaches        -> ./data_prepared/awardsCoaches.csv  (29 rows, 5 cols)
  saved awardsPlayers        -> ./data_prepared/awardsPlayers.csv  (930 rows, 6 cols)
  saved coaches              -> ./data_prepared/coaches.csv  (892 rows, 14 cols)
  saved combinedShutouts     -> ./data_prepared/combinedShutouts.csv  (36 rows, 8 cols)
  saved goalies              -> ./data_prepared/goalies.csv  (2405 rows, 23 cols)
  saved goaliesShootout      -> ./data_prepared/goaliesShootout.csv  (346 rows, 8 cols)
  saved hof                  -> ./data_prepared/hof.csv  (131 rows, 4 cols)
  saved master               -> ./data_prepared/master.csv  (7469 rows, 31 cols)
  saved scoring              -> ./data_prepared/scoring.csv  (26497 rows, 31 cols)
  saved scoringShootout      -> ./data_prepared/scoringShootout.csv  (1487 rows, 7 cols)
  saved seriesPost           -> ./data_prepared/seriesPost.csv  (435 rows, 13 cols)